<a href="https://colab.research.google.com/github/manaswiarvind/AI-LAB/blob/main/resolution%20fol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

def parse_literal(lit_str):
    lit_str = lit_str.strip()
    neg = lit_str.startswith('~') or lit_str.startswith('¬')
    if neg:
        lit_str = lit_str[1:].strip()
    match = re.match(r'^([A-Za-z0-9_]+)\((.*)\)$', lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: {lit_str}")
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]
    return (neg, pred, tuple(args))

def format_literal(lit):
    neg, pred, args = lit
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    if not clause:
        return "{} (Empty Clause - Contradiction)"
    return " V ".join(format_literal(lit) for lit in clause)

def is_variable(term):
    return term[0].islower() and '(' not in term

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is None:
        return None
    if x == y:
        return subst
    if isinstance(x, str) and is_variable(x):
        return unify_var(x, y, subst)
    if isinstance(y, str) and is_variable(y):
        return unify_var(y, x, subst)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst
    return None

def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    if isinstance(x, str) and x in subst:
        return unify(var, subst[x], subst)
    if occur_check(var, x, subst):
        return None
    new_subst = subst.copy()
    new_subst[var] = x
    return new_subst

def occur_check(var, x, subst):
    if var == x:
        return True
    if isinstance(x, str) and x in subst:
        return occur_check(var, subst[x], subst)
    if isinstance(x, tuple):
        return any(occur_check(var, elem, subst) for elem in x)
    return False

def apply_subst_term(term, subst):
    while isinstance(term, str) and term in subst:
        term = subst[term]
    return term

def apply_subst_literal(lit, subst):
    neg, pred, args = lit
    new_args = tuple(apply_subst_term(arg, subst) for arg in args)
    return (neg, pred, new_args)

def resolve_clauses(c1, c2):
    resolvents = []
    for l1 in c1:
        for l2 in c2:
            # Check for complementary predicates
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                subst = unify(l1[2], l2[2], {})
                if subst is not None:
                    new_c1 = [apply_subst_literal(l, subst) for l in c1 if l != l1]
                    new_c2 = [apply_subst_literal(l, subst) for l in c2 if l != l2]
                    # Remove duplicate literals while preserving order
                    combined = []
                    for l in new_c1 + new_c2:
                        if l not in combined:
                            combined.append(l)
                    resolvents.append((tuple(combined), subst))
    return resolvents

def fol_resolution(kb_clauses, negated_query_clauses):
    clauses = list(kb_clauses) + list(negated_query_clauses)
    seen = set(frozenset(c) for c in clauses)

    print("\n--- Initial Clauses (KB + Negated Query) ---")
    for idx, c in enumerate(clauses, 1):
        print(f"{idx}. {format_clause(c)}")
    print("\n--- Resolution Steps ---")

    step = 1
    while True:
        new_clauses = []
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve_clauses(clauses[i], clauses[j])
                for res, subst in resolvents:
                    res_set = frozenset(res)
                    if not res:
                        print(f"Step {step}: Resolve [{format_clause(clauses[i])}] and [{format_clause(clauses[j])}]")
                        if subst:
                            print(f"        Unification: {subst}")
                        print(f"        Resolvent: {format_clause(res)}")
                        return True
                    if res_set not in seen:
                        seen.add(res_set)
                        new_clauses.append(res)
                        print(f"Step {step}: Resolve [{format_clause(clauses[i])}] and [{format_clause(clauses[j])}]")
                        if subst:
                            print(f"        Unification: {subst}")
                        print(f"        Resolvent: {format_clause(res)}")
                        step += 1
        if not new_clauses:
            return False
        clauses.extend(new_clauses)

def main():
    print("=== First-Order Logic (FOL) Resolution Prover ===")
    print("Formatting Rules:")
    print(" - Use lowercase for variables (e.g., x, y, z)")
    print(" - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))")
    print(" - Use '~' for negation and 'V' for OR (e.g., ~Food(x) V Likes(John, x))\n")

    num_clauses = int(input("Enter the number of CNF clauses in the Knowledge Base: "))
    kb = []
    for i in range(num_clauses):
        raw = input(f"Clause {i+1}: ")
        literals = [parse_literal(part) for part in raw.split('V')]
        kb.append(tuple(literals))

    query_raw = input("\nEnter the single atomic query to prove (e.g., Likes(John, Peanuts)): ")
    q_neg, q_pred, q_args = parse_literal(query_raw)
    # Negate the query for proof by refutation
    negated_query = [((not q_neg, q_pred, q_args),)]

    proved = fol_resolution(kb, negated_query)
    print("\n=== Result ===")
    if proved:
        print(f"Hence Proved: {query_raw} follows from the Knowledge Base.")
    else:
        print(f"Cannot Prove: {query_raw} does not follow from the Knowledge Base.")

if __name__ == "__main__":
    main()

=== First-Order Logic (FOL) Resolution Prover ===
Formatting Rules:
 - Use lowercase for variables (e.g., x, y, z)
 - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))
 - Use '~' for negation and 'V' for OR (e.g., ~Food(x) V Likes(John, x))

Enter the number of CNF clauses in the Knowledge Base: 9
Clause 1:  ~Food(x) V Likes(John, x)
Clause 2: Food(Apple)
Clause 3: Food(Vegetables)


ValueError: Invalid literal format: Food(

In [3]:
import re

def parse_literal(lit_str):
    lit_str = lit_str.strip()
    neg = lit_str.startswith('~') or lit_str.startswith('¬')
    if neg:
        lit_str = lit_str[1:].strip()
    match = re.match(r'^([A-Za-z0-9_]+)\((.*)\)$', lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: {lit_str}")
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]
    return (neg, pred, tuple(args))

def format_literal(lit):
    neg, pred, args = lit
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    if not clause:
        return "{} (Empty Clause - Contradiction)"
    return " V ".join(format_literal(lit) for lit in clause)

def is_variable(term):
    return term.islower() and '(' not in term

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if x == y:
        return subst
    if isinstance(x, str) and is_variable(x):
        return unify_var(x, y, subst)
    if isinstance(y, str) and is_variable(y):
        return unify_var(y, x, subst)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst
    return None

def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    if isinstance(x, str) and x in subst:
        return unify(var, subst[x], subst)
    if occur_check(var, x, subst):
        return None
    new_subst = subst.copy()
    new_subst[var] = x
    return new_subst

def occur_check(var, x, subst):
    if var == x:
        return True
    if isinstance(x, str) and x in subst:
        return occur_check(var, subst[x], subst)
    if isinstance(x, tuple):
        return any(occur_check(var, elem, subst) for elem in x)
    return False

def apply_subst_term(term, subst):
    while isinstance(term, str) and term in subst:
        term = subst[term]
    return term

def apply_subst_literal(lit, subst):
    neg, pred, args = lit
    new_args = tuple(apply_subst_term(arg, subst) for arg in args)
    return (neg, pred, new_args)

def resolve_clauses(c1, c2):
    resolvents = []
    for l1 in c1:
        for l2 in c2:
            # Check opposite signs, same predicate name, and same number of arguments
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                subst = unify(l1[2], l2[2], {})
                if subst is not None:
                    new_c1 = [apply_subst_literal(l, subst) for l in c1 if l != l1]
                    new_c2 = [apply_subst_literal(l, subst) for l in c2 if l != l2]
                    combined = []
                    for l in new_c1 + new_c2:
                        if l not in combined:
                            combined.append(l)
                    resolvents.append((tuple(combined), subst))
    return resolvents

def fol_resolution(kb_clauses, negated_query_clauses):
    clauses = list(kb_clauses) + list(negated_query_clauses)
    seen = set(frozenset(c) for c in clauses)

    print("\n--- Initial Clauses (KB + Negated Query) ---")
    for idx, c in enumerate(clauses, 1):
        print(f"{idx}. {format_clause(c)}")
    print("\n--- Resolution Steps ---")

    step = 1
    while True:
        new_clauses = []
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve_clauses(clauses[i], clauses[j])
                for res, subst in resolvents:
                    res_set = frozenset(res)
                    if not res:
                        print(f"Step {step}: Resolve [{format_clause(clauses[i])}] and [{format_clause(clauses[j])}]")
                        if subst:
                            print(f"        Unification: {subst}")
                        print(f"        Resolvent: {format_clause(res)}")
                        return True
                    if res_set not in seen:
                        seen.add(res_set)
                        new_clauses.append(res)
                        print(f"Step {step}: Resolve [{format_clause(clauses[i])}] and [{format_clause(clauses[j])}]")
                        if subst:
                            print(f"        Unification: {subst}")
                        print(f"        Resolvent: {format_clause(res)}")
                        step += 1
        if not new_clauses:
            return False
        clauses.extend(new_clauses)

def main():
    print("=== First-Order Logic (FOL) Resolution Prover ===")
    print("Formatting Rules:")
    print(" - Use lowercase for variables (e.g., x, y, z)")
    print(" - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))")
    print(" - Use '~' for negation and ' | ' or ' V ' (with spaces) for OR\n")

    num_clauses = int(input("Enter the number of CNF clauses in the Knowledge Base: "))
    kb = []
    for i in range(num_clauses):
        raw = input(f"Clause {i+1}: ")
        # Split on '|' or ' V ' with spaces so words like 'Vegetables' are not split
        parts = re.split(r'\s*\|\s*|\s+V\s+', raw.strip())
        literals = [parse_literal(part) for part in parts if part]
        kb.append(tuple(literals))

    query_raw = input("\nEnter the single atomic query to prove (e.g., Likes(John, Peanuts)): ")
    q_neg, q_pred, q_args = parse_literal(query_raw)
    negated_query = [((not q_neg, q_pred, q_args),)]

    proved = fol_resolution(kb, negated_query)
    print("\n=== Result ===")
    if proved:
        print(f"Hence Proved: {query_raw} follows from the Knowledge Base.")
    else:
        print(f"Cannot Prove: {query_raw} does not follow from the Knowledge Base.")

if __name__ == "__main__":
    main()


=== First-Order Logic (FOL) Resolution Prover ===
Formatting Rules:
 - Use lowercase for variables (e.g., x, y, z)
 - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))
 - Use '~' for negation and ' | ' or ' V ' (with spaces) for OR

Enter the number of CNF clauses in the Knowledge Base: 9
Clause 1: ~Food(x) V Likes(John, x)
Clause 2: Food(Apple)
Clause 3: Food(Vegetables)
Clause 4: ~Eats(y, z) V Killed(y) V Food(z)
Clause 5: Eats(Anil, Peanuts)
Clause 6: Alive(Anil)
Clause 7: ~Eats(Anil, w) V Eats(Harry, w)
Clause 8: Killed(g) V Alive(g)
Clause 9: ~Alive(k) V ~Killed(k)

Enter the single atomic query to prove (e.g., Likes(John, Peanuts)): Likes(John, Peanuts)

--- Initial Clauses (KB + Negated Query) ---
1. ~Food(x) V Likes(John, x)
2. Food(Apple)
3. Food(Vegetables)
4. ~Eats(y, z) V Killed(y) V Food(z)
5. Eats(Anil, Peanuts)
6. Alive(Anil)
7. ~Eats(Anil, w) V Eats(Harry, w)
8. Killed(g) V Alive(g)
9. ~Alive(k) V ~Killed(k)
10. ~Likes(John, Peanuts)

--- Resolut

In [4]:
import re

def parse_literal(lit_str):
    lit_str = lit_str.strip()
    neg = lit_str.startswith('~') or lit_str.startswith('¬')
    if neg:
        lit_str = lit_str[1:].strip()
    match = re.match(r'^([A-Za-z0-9_]+)\((.*)\)$', lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: {lit_str}")
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]
    return (neg, pred, tuple(args))

def format_literal(lit):
    neg, pred, args = lit
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    if not clause:
        return "{}  (Hence Proved)"
    return " V ".join(format_literal(lit) for lit in clause)

def format_subst(subst):
    if not subst:
        return ""
    pairs = [f"{val}/{var}" for var, val in subst.items()]
    return "{" + ", ".join(pairs) + "}"

def is_variable(term):
    return term.islower() and '(' not in term

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if x == y:
        return subst
    if isinstance(x, str) and is_variable(x):
        return unify_var(x, y, subst)
    if isinstance(y, str) and is_variable(y):
        return unify_var(y, x, subst)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst
    return None

def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    if isinstance(x, str) and x in subst:
        return unify(var, subst[x], subst)
    if occur_check(var, x, subst):
        return None
    new_subst = subst.copy()
    new_subst[var] = x
    return new_subst

def occur_check(var, x, subst):
    if var == x:
        return True
    if isinstance(x, str) and x in subst:
        return occur_check(var, subst[x], subst)
    if isinstance(x, tuple):
        return any(occur_check(var, elem, subst) for elem in x)
    return False

def apply_subst_term(term, subst):
    while isinstance(term, str) and term in subst:
        term = subst[term]
    return term

def apply_subst_literal(lit, subst):
    neg, pred, args = lit
    new_args = tuple(apply_subst_term(arg, subst) for arg in args)
    return (neg, pred, new_args)

def resolve_clauses(c1, c2):
    resolvents = []
    for l1 in c1:
        for l2 in c2:
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                subst = unify(l1[2], l2[2], {})
                if subst is not None:
                    new_c1 = [apply_subst_literal(l, subst) for l in c1 if l != l1]
                    new_c2 = [apply_subst_literal(l, subst) for l in c2 if l != l2]
                    combined = []
                    for l in new_c1 + new_c2:
                        if l not in combined:
                            combined.append(l)
                    resolvents.append((tuple(combined), subst))
    return resolvents

def collect_proof_steps(nodes, node_idx, steps):
    node = nodes[node_idx]
    if node["parents"] is None:
        return
    p1, p2 = node["parents"]
    # Recursively visit deeper derived parent first so tree flows top-down
    if nodes[p1]["parents"] is not None:
        collect_proof_steps(nodes, p1, steps)
        left_idx, right_idx = p1, p2
    else:
        collect_proof_steps(nodes, p2, steps)
        left_idx, right_idx = p2, p1
    steps.append((left_idx, right_idx, node_idx))

def print_top_down_tree(nodes, goal_idx):
    steps = []
    collect_proof_steps(nodes, goal_idx, steps)

    print("\n=== Top-Down Resolution Proof Tree ===\n")
    indent = ""
    for step_num, (left_idx, right_idx, res_idx) in enumerate(steps):
        left_node = nodes[left_idx]
        right_node = nodes[right_idx]
        res_node = nodes[res_idx]

        left_str = format_clause(left_node["clause"])
        right_str = f"[{right_node['source']}] {format_clause(right_node['clause'])}"
        subst_str = format_subst(res_node["subst"])
        res_str = format_clause(res_node["clause"])

        if step_num == 0:
            left_str = f"[{left_node['source']}] {left_str}"

        width = max(len(left_str) + len(right_str) + 6, 40)
        print(f"{indent}{left_str:<28} {right_str}")
        if subst_str:
            print(f"{indent}{' ' * 24}{subst_str}")
        print(f"{indent}   \\ {' ' * 22} /")
        print(f"{indent}    \\{'_' * 22}/")
        print(f"{indent}             |")
        indent += "      "
        if step_num == len(steps) - 1:
            print(f"{indent}   {res_str}")

def fol_resolution(kb_clauses, negated_query_clauses):
    nodes = []
    seen = set()

    for idx, c in enumerate(kb_clauses, 1):
        nodes.append({"clause": c, "parents": None, "subst": None, "source": f"KB-{idx}"})
        seen.add(frozenset(c))

    for idx, c in enumerate(negated_query_clauses, 1):
        nodes.append({"clause": c, "parents": None, "subst": None, "source": "Negated Query"})
        seen.add(frozenset(c))

    print("\n--- Initial Clauses ---")
    for idx, node in enumerate(nodes, 1):
        print(f"{idx}. [{node['source']}] {format_clause(node['clause'])}")

    while True:
        new_nodes = []
        n = len(nodes)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve_clauses(nodes[i]["clause"], nodes[j]["clause"])
                for res, subst in resolvents:
                    res_set = frozenset(res)
                    if not res:
                        goal_node = {"clause": res, "parents": (i, j), "subst": subst, "source": "Resolvent"}
                        nodes.append(goal_node)
                        print_top_down_tree(nodes, len(nodes) - 1)
                        return True
                    if res_set not in seen:
                        seen.add(res_set)
                        new_nodes.append({"clause": res, "parents": (i, j), "subst": subst, "source": "Resolvent"})
        if not new_nodes:
            return False
        nodes.extend(new_nodes)

def main():
    print("=== First-Order Logic (FOL) Resolution Prover ===")
    print("Formatting Rules:")
    print(" - Use lowercase for variables (e.g., x, y, z)")
    print(" - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))")
    print(" - Use '~' for negation and ' | ' or ' V ' (with spaces) for OR\n")

    num_clauses = int(input("Enter the number of CNF clauses in the Knowledge Base: "))
    kb = []
    for i in range(num_clauses):
        raw = input(f"Clause {i+1}: ")
        parts = re.split(r'\s*\|\s*|\s+V\s+', raw.striAp())
        literals = [parse_literal(part) for part in parts if part]
        kb.append(tuple(literals))

    query_raw = input("\nEnter the single atomic query to prove (e.g., Likes(John, Peanuts)): ")
    q_neg, q_pred, q_args = parse_literal(query_raw)
    negated_query = [((not q_neg, q_pred, q_args),)]

    proved = fol_resolution(kb, negated_query)
    print("\n=== Result ===")
    if proved:
        print(f"Hence Proved: {query_raw} follows from the Knowledge Base.")
    else:
        print(f"Cannot Prove: {query_raw} does not follow from the Knowledge Base.")

if __name__ == "__main__":
    main()



=== First-Order Logic (FOL) Resolution Prover ===
Formatting Rules:
 - Use lowercase for variables (e.g., x, y, z)
 - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))
 - Use '~' for negation and ' | ' or ' V ' (with spaces) for OR

Enter the number of CNF clauses in the Knowledge Base: 9
Clause 1: ~Food(x) V Likes(John, x)
Clause 2: Food(Apple)
Clause 3: Food(Vegetables)
Clause 4: ~Eats(y, z) V Killed(y) V Food(z)
Clause 5:  Eats(Anil, Peanuts)
Clause 6: Alive(Anil)
Clause 7: ~Eats(Anil, w) V Eats(Harry, w)
Clause 8: Killed(g) V Alive(g)
Clause 9: ~Alive(k) V ~Killed(k)

Enter the single atomic query to prove (e.g., Likes(John, Peanuts)): Likes(John, Peanuts)

--- Initial Clauses ---
1. [KB-1] ~Food(x) V Likes(John, x)
2. [KB-2] Food(Apple)
3. [KB-3] Food(Vegetables)
4. [KB-4] ~Eats(y, z) V Killed(y) V Food(z)
5. [KB-5] Eats(Anil, Peanuts)
6. [KB-6] Alive(Anil)
7. [KB-7] ~Eats(Anil, w) V Eats(Harry, w)
8. [KB-8] Killed(g) V Alive(g)
9. [KB-9] ~Alive(k) V ~Kille

In [7]:
import re

def parse_literal(lit_str):
    lit_str = lit_str.strip()
    neg = lit_str.startswith('~') or lit_str.startswith('¬')
    if neg:
        lit_str = lit_str[1:].strip()
    match = re.match(r'^([A-Za-z0-9_]+)\((.*)\)$', lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: {lit_str}")
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]
    return (neg, pred, tuple(args))

def format_literal(lit):
    neg, pred, args = lit
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    if not clause:
        return "{}  (Hence Proved)"
    return " V ".join(format_literal(lit) for lit in clause)

def format_subst(subst):
    if not subst:
        return ""
    pairs = [f"{val}/{var}" for var, val in subst.items()]
    return "{" + ", ".join(pairs) + "}"

def is_variable(term):
    return term.islower() and '(' not in term

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if x == y:
        return subst
    if isinstance(x, str) and is_variable(x):
        return unify_var(x, y, subst)
    if isinstance(y, str) and is_variable(y):
        return unify_var(y, x, subst)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst
    return None

def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    if isinstance(x, str) and x in subst:
        return unify(var, subst[x], subst)
    if occur_check(var, x, subst):
        return None
    new_subst = subst.copy()
    new_subst[var] = x
    return new_subst

def occur_check(var, x, subst):
    if var == x:
        return True
    if isinstance(x, str) and x in subst:
        return occur_check(var, subst[x], subst)
    if isinstance(x, tuple):
        return any(occur_check(var, elem, subst) for elem in x)
    return False

def apply_subst_term(term, subst):
    while isinstance(term, str) and term in subst:
        term = subst[term]
    return term

def apply_subst_literal(lit, subst):
    neg, pred, args = lit
    new_args = tuple(apply_subst_term(arg, subst) for arg in args)
    return (neg, pred, new_args)

def resolve_clauses(c1, c2):
    resolvents = []
    for l1 in c1:
        for l2 in c2:
            # Check opposite negation, same predicate name, and same argument count
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                subst = unify(l1[2], l2[2], {})
                if subst is not None:
                    new_c1 = [apply_subst_literal(l, subst) for l in c1 if l != l1]
                    new_c2 = [apply_subst_literal(l, subst) for l in c2 if l != l2]
                    combined = []
                    for l in new_c1 + new_c2:
                        if l not in combined:
                            combined.append(l)
                    resolvents.append((tuple(combined), subst))
    return resolvents

def collect_proof_steps(nodes, node_idx, steps):
    node = nodes[node_idx]
    if node["parents"] is None:
        return
    p1, p2 = node["parents"]
    if nodes[p1]["parents"] is not None:
        collect_proof_steps(nodes, p1, steps)
        left_idx, right_idx = p1, p2
    else:
        collect_proof_steps(nodes, p2, steps)
        left_idx, right_idx = p2, p1
    steps.append((left_idx, right_idx, node_idx))

def print_top_down_tree(nodes, goal_idx):
    steps = []
    collect_proof_steps(nodes, goal_idx, steps)

    print("\n=== Top-Down Resolution Proof Tree ===\n")
    indent = ""
    for step_num, (left_idx, right_idx, res_idx) in enumerate(steps):
        left_node = nodes[left_idx]
        right_node = nodes[right_idx]
        res_node = nodes[res_idx]

        left_str = format_clause(left_node["clause"])
        right_str = f"[{right_node['source']}] {format_clause(right_node['clause'])}"
        subst_str = format_subst(res_node["subst"])
        res_str = format_clause(res_node["clause"])

        if step_num == 0:
            left_str = f"[{left_node['source']}] {left_str}"

        print(f"{indent}{left_str:<32} {right_str}")
        if subst_str:
            print(f"{indent}{' ' * 26}{subst_str}")
        print(f"{indent}   \\ {' ' * 24} /")
        print(f"{indent}    \\{'_' * 24}/")
        print(f"{indent}              |")
        indent += "      "
        if step_num == len(steps) - 1:
            print(f"{indent}   {res_str}")

def fol_resolution(kb_clauses, negated_query_clauses):
    nodes = []
    seen = set()

    for idx, c in enumerate(kb_clauses, 1):
        nodes.append({"clause": c, "parents": None, "subst": None, "source": f"KB-{idx}"})
        seen.add(frozenset(c))

    for idx, c in enumerate(negated_query_clauses, 1):
        nodes.append({"clause": c, "parents": None, "subst": None, "source": "Negated Query"})
        seen.add(frozenset(c))

    print("\n--- Initial Clauses ---")
    for idx, node in enumerate(nodes, 1):
        print(f"{idx}. [{node['source']}] {format_clause(node['clause'])}")

    while True:
        new_nodes = []
        n = len(nodes)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve_clauses(nodes[i]["clause"], nodes[j]["clause"])
                for res, subst in resolvents:
                    res_set = frozenset(res)
                    if not res:
                        goal_node = {"clause": res, "parents": (i, j), "subst": subst, "source": "Resolvent"}
                        nodes.append(goal_node)
                        print_top_down_tree(nodes, len(nodes) - 1)
                        return True
                    if res_set not in seen:
                        seen.add(res_set)
                        new_nodes.append({"clause": res, "parents": (i, j), "subst": subst, "source": "Resolvent"})
        if not new_nodes:
            return False
        nodes.extend(new_nodes)

def main():
    print("=== First-Order Logic (FOL) Resolution Prover ===")
    print("Formatting Rules:")
    print(" - Use lowercase for variables (e.g., x, y, z)")
    print(" - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))")
    print(" - Use '~' for negation and ' | ' or ' V ' (with spaces) for OR\n")

    num_clauses = int(input("Enter the number of CNF clauses in the Knowledge Base: "))
    kb = []
    for i in range(num_clauses):
        raw = input(f"Clause {i+1}: ")
        parts = re.split(r'\s*\|\s*|\s+V\s+', raw.strip())
        literals = [parse_literal(part) for part in parts if part]
        kb.append(tuple(literals))

    query_raw = input("\nEnter the single atomic query to prove (e.g., Sneezes(Mary)): ")
    q_neg, q_pred, q_args = parse_literal(query_raw)
    negated_query = [((not q_neg, q_pred, q_args),)]

    proved = fol_resolution(kb, negated_query)
    print("\n=== Result ===")
    if proved:
        print(f"Hence Proved: {query_raw} follows from the Knowledge Base.")
    else:
        print(f"Cannot Prove: {query_raw} does not follow from the Knowledge Base.")

if __name__ == "__main__":
    main()


=== First-Order Logic (FOL) Resolution Prover ===
Formatting Rules:
 - Use lowercase for variables (e.g., x, y, z)
 - Use Uppercase for constants and predicates (e.g., John, Peanuts, Food(x))
 - Use '~' for negation and ' | ' or ' V ' (with spaces) for OR

Enter the number of CNF clauses in the Knowledge Base: 5
Clause 1: ~Allergies(z) V Sneezes(z)
Clause 2: ~LivesWith(x, y) V ~Cat(y) V ~AllergicToCats(x) V Allergies(x)
Clause 3:  Cat(Tom)
Clause 4: AllergicToCats(Mary)
Clause 5: LivesWith(Mary, Tom)

Enter the single atomic query to prove (e.g., Sneezes(Mary)): Sneezes(Mary)

--- Initial Clauses ---
1. [KB-1] ~Allergies(z) V Sneezes(z)
2. [KB-2] ~LivesWith(x, y) V ~Cat(y) V ~AllergicToCats(x) V Allergies(x)
3. [KB-3] Cat(Tom)
4. [KB-4] AllergicToCats(Mary)
5. [KB-5] LivesWith(Mary, Tom)
6. [Negated Query] ~Sneezes(Mary)

=== Top-Down Resolution Proof Tree ===

[Negated Query] ~Sneezes(Mary)   [KB-1] ~Allergies(z) V Sneezes(z)
                          {Mary/z}
   \                    